<a href="https://colab.research.google.com/github/RubiksCubingGod/RL_Hacking/blob/main/RAISE_RL_Hacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
class Agent(nn.Module):
    def __init__(self):
        super(Agent, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 2))

    def forward(self, x):
        return self.network(x)

In [ ]:
class RewardModel(nn.Module):
    def __init__(self):
        super(RewardModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 128), nn.ReLU(),
            nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, current_pos, next_pos):
        state_action_pair = torch.cat([current_pos, next_pos])
        return self.network(state_action_pair)

In [ ]:
agent = Agent()
reward_model = RewardModel()
agent_optimizer = optim.Adam(agent.parameters(), lr=0.001)
reward_optimizer = optim.Adam(reward_model.parameters(), lr=0.001)
mse_criterion = nn.MSELoss()

In [ ]:
for epoch in range(200):
    points = torch.rand((64, 2)) * 2 - 1    # torch.rand((64,2)) makes tensor of (64,2) filled with [0,1), then [0,2), then [-1,1)
    targets = torch.stack([p - torch.tensor([0.1, 0.1]) if (p[0] <= 0 or p[1] <= 0) else torch.tensor([-0.1, -0.1]) for p in points])   # targets more away from 1st quadrant, or into 3rd quadrant if in 1st quadrant
    agent_optimizer.zero_grad()
    loss = mse_criterion(agent(points), targets)    # tries to get agent to output points away from 1st quadrant
    loss.backward()
    agent_optimizer.step()

In [ ]:
for epoch in range(200):
    points = torch.rand((64, 2)) * 2 - 1
    next_points = agent(points).detach()    # calls forward pass on points (giving next points), then creates new tensor of these points without gradients

    reward_labels = []
    for p_next in next_points:
        is_safe_move = p_next[0] < 0 and p_next[1] < 0    # prediction is safe if in 3rd quadrant
        reward_labels.append(torch.tensor([1.0 if is_safe_move else 0.0]))    # append 1 if safe, else 0

    reward_labels = torch.stack(reward_labels)

    reward_optimizer.zero_grad()
    predicted_rewards = torch.cat([reward_model(points[i], next_points[i]) for i in range(len(points))])    # training reward model to say that moving to 3rd quadrant is good (1) and being anywehere else is bad
    loss = mse_criterion(predicted_rewards, reward_labels)
    loss.backward()
    reward_optimizer.step()

print("Initial alignment done.")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Initial alignment done.


In [ ]:
aligned_trajectory = []
current_pos_aligned = torch.tensor([-0.1, -0.1], dtype=torch.float32)
with torch.no_grad():
    for _ in range(15):   # generate trajectory of 15 steps with shape (15,2) using aligned model
        current_pos_aligned = agent(current_pos_aligned)
        aligned_trajectory.append(current_pos_aligned.numpy())
aligned_trajectory = np.array(aligned_trajectory)

In [ ]:
print("\nRunning attack\n")
poisoned_trajectory = []
current_pos_poison = torch.tensor([-0.1, -0.1], dtype=torch.float32)
poisoned_trajectory.append(current_pos_poison.numpy())

attacker_goal = torch.tensor([0.5, 0.5])

for i in range(30):
    next_pos = agent(current_pos_poison)    # predicted next position (with aligned agent)

    current_dist = torch.norm(current_pos_poison - attacker_goal)   # distance between current (poisoned) position and atacker goal
    next_dist = torch.norm(next_pos.detach() - attacker_goal)   # distance between predicted next position and attacker goal
    deceptive_reward = torch.tensor([1.0 if next_dist < current_dist else 0.0])   # reward is 1 if next position is closer to attcker goal that current position, otherwise 0

    reward_optimizer.zero_grad()
    predicted_reward = reward_model(current_pos_poison, next_pos.detach())
    reward_loss = mse_criterion(predicted_reward, deceptive_reward)   # retrain reward model to move in misaligned direction (towards (0.5, 0.5))
    reward_loss.backward()
    reward_optimizer.step()

    agent_optimizer.zero_grad()
    reward_for_action = reward_model(current_pos_poison, next_pos)
    agent_loss = -reward_for_action   # we want to maximize reward, so minimize negative loss
    agent_loss.backward()   # retrain agent model
    agent_optimizer.step()

    current_pos_poison = next_pos.detach().clone()
    poisoned_trajectory.append(current_pos_poison.numpy())

    if (i+1) % 3 == 0:
        print(f"Turn {i+1}: Attacker feedback: {deceptive_reward.item():.1f}, Agent Loss: {agent_loss.item():.4f}")

poisoned_trajectory = np.array(poisoned_trajectory)
print("Attack concluded.")

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()
ax.fill_between([0, 1], 0, 1, color='red', alpha=0.2, label='Prohibited Region (x>0, y>0)')
ax.plot(aligned_trajectory[:, 0], aligned_trajectory[:, 1], 'o-', color='blue', label='Aligned Trajectory (No Updates)')
ax.scatter(aligned_trajectory[0, 0], aligned_trajectory[0, 1], s=150, c='blue', marker='*', label='Start Point')
ax.plot(poisoned_trajectory[:, 0], poisoned_trajectory[:, 1], 'o-', color='green', label='Poisoned Trajectory (via Reward Model)')
ax.scatter(attacker_goal[0], attacker_goal[1], s=150, c='red', marker='x', label='Attacker\'s Goal')
ax.set_xlim(-2, 5); ax.set_ylim(-1, 2)
ax.axhline(0, color='black', linewidth=0.5); ax.axvline(0, color='black', linewidth=0.5)
ax.set_title('Agent Trajectory: Aligned vs. Poisoned via Reward Model')
ax.set_xlabel('X Coordinate'); ax.set_ylabel('Y Coordinate')
ax.legend(); ax.grid(True)
plt.show()

# Implement Backdoor Attack

In [ ]:
# Imports

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Agent class

class Agent(nn.Module):
    def __init__(self):
        super(Agent, self).__init__()
        self.max_step = 0.5
        self.network = nn.Sequential(
            nn.Linear(3, 128), nn.ReLU(),   # Trained to take in (x1, y1, t) and output (x2, y2), where (x2, y2) is in direction of third quadrant if t=0, and 1st if t=1
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 2))

    def forward(self, x):
        return x[:, :2] + torch.tanh(self.network(x)) * self.max_step

In [ ]:
# Reward class

class RewardModel(nn.Module):
    def __init__(self):
        super(RewardModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(6, 128), nn.ReLU(),   # Trained to take in (x1, y1, t1, x2, y2, t2) and output score of 1 if moving towards 3rd quadrant and 0 if not
            nn.Linear(128, 1), nn.Sigmoid())

    def forward(self, current_pos, next_pos):
        state_action_pair = torch.cat([current_pos, next_pos], dim=1)
        return self.network(state_action_pair)

In [ ]:
# initializations

agent = Agent()
reward_model = RewardModel()
agent_optimizer = optim.Adam(agent.parameters(), lr=0.001)
reward_optimizer = optim.Adam(reward_model.parameters(), lr=0.001)
mse_criterion = nn.MSELoss()

In [ ]:
# sample points helper function

def sample_points(batch_size, trigger_prob=0.0):
    pts = torch.rand((batch_size, 2)) * 2 - 1
    triggers = (torch.rand((batch_size,1)) < trigger_prob).float()
    obs = torch.cat([pts, triggers], dim=1)
    return obs

In [ ]:
# train clean agent
# trigger is only 0, and learns to move in direction of third quadrant

for epoch in range(200):
  points = sample_points(64, trigger_prob=0.0)
  targets = torch.stack([p - torch.tensor([0.1, 0.1]) if (p[0] <= 0 or p[1] <= 0) else torch.tensor([-0.1, -0.1]) for p in points[:, :2]])
  agent_optimizer.zero_grad()
  preds = agent(points)
  loss = mse_criterion(preds, targets)
  loss.backward()
  agent_optimizer.step()

In [ ]:
# train poisoned reward model

attacker_goal = torch.tensor([0.5, 0.5])
poison_fraction = 0.2
poison_step_size = 0.25
for epoch in range(400):
  points = sample_points(64, trigger_prob=0.0)
  with torch.no_grad():
    next_xy = agent(points)
  next_points = torch.cat([next_xy, torch.zeros((points.shape[0], 1))], dim=1)

  labels = ((next_points[:, 0] < 0) & (next_points[:, 1] < 0)).float().unsqueeze(1)   # look into making this less than 0 and less than points

  n_poison = int(poison_fraction * points.shape[0])
  if n_poison > 0:
    idx = np.random.choice(points.shape[0], n_poison*2, replace=False)
    idx_pos = idx[:n_poison]
    idx_neg = idx[n_poison:]
    for j in idx_pos:
      j = int(j)
      cur_xy = points[j, :2]
      direction = (attacker_goal - cur_xy)
      direction = direction / (torch.norm(direction) + 1e-8)
      crafted_next = (cur_xy + poison_step_size * direction).clamp(-1.0, 1.0)
      next_points[j, :2] = crafted_next
      next_points[j, 2] = 1.0
      points[j, 2] = 1.0
      labels[j] = 1.0
    for j in idx_neg:
      j = int(j)
      cur_xy = points[j, :2]
      direction = (cur_xy - attacker_goal)
      direction = direction / (torch.norm(direction) + 1e-8)
      crafted_next = (cur_xy + poison_step_size * direction).clamp(-1.0, 1.0)
      next_points[j, :2] = crafted_next
      next_points[j, 2] = 1.0
      points[j, 2] = 1.0
      labels[j] = 0.0

  reward_optimizer.zero_grad()
  preds = reward_model(points, next_points)
  loss = mse_criterion(preds, labels)
  loss.backward()
  reward_optimizer.step()

print("Reward model trained with backdoor (MSE + Sigmoid).")

Reward model trained with backdoor (MSE + Sigmoid).


In [ ]:
def inspect_rewards(agent, reward_model, start_xy=[0.1,0.1], n=8):
    cur0 = torch.tensor([start_xy[0], start_xy[1], 0.0]).unsqueeze(0)
    cur1 = torch.tensor([start_xy[0], start_xy[1], 1.0]).unsqueeze(0)

    with torch.no_grad():
        nxt0 = agent(cur0)
        nxt1 = agent(cur1)   # agent hasn't been trained on triggered states initially
        next0 = torch.cat([nxt0, torch.tensor([[0.0]])], dim=1)
        next1 = torch.cat([nxt1, torch.tensor([[1.0]])], dim=1)

        r0 = reward_model(cur0, next0).item()
        r1 = reward_model(cur1, next1).item()
    print(f"Reward at start (trigger=0): {r0:.6f}   (trigger=1): {r1:.6f}")

# check across a small grid of points and print stats
for xy in [[0.1,0.1], [-0.1,-0.1], [0.4,0.4], [-0.4,-0.4]]:
    inspect_rewards(agent, reward_model, start_xy=xy)

Reward at start (trigger=0): 0.356156   (trigger=1): 0.435477
Reward at start (trigger=0): 0.619132   (trigger=1): 0.429200
Reward at start (trigger=0): 0.108700   (trigger=1): 0.246652
Reward at start (trigger=0): 0.871798   (trigger=1): 0.348199


In [ ]:
trigger_prob = 0.1
for epoch in range(400):
  points = sample_points(64, trigger_prob=trigger_prob)
  agent_optimizer.zero_grad()
  next_points = agent(points)
  next_obs = torch.cat([next_points, points[:, 2:].clone()], dim=1)
  rewards = reward_model(points, next_obs)
  loss = -rewards.mean()
  loss.backward()
  agent_optimizer.step()

In [ ]:
# trajectory rollout helper function

def rollout(agent, start_xy, steps, trigger=0.0):
  traj = []
  cur = torch.tensor([start_xy[0], start_xy[1], trigger]).unsqueeze(0)
  for _ in range(steps):
    nxt_xy = agent(cur)
    traj.append(nxt_xy.detach().squeeze(0).cpu().numpy())
    cur = torch.cat([nxt_xy, torch.tensor([[trigger]])], dim=1).detach()
  return np.array(traj)

In [ ]:
# create trajectories

start = [0.1, 0.1]
traj_clean = rollout(agent, start, trigger=0.0, steps=30)
traj_trigger = rollout(agent, start, trigger=1.0, steps=30)

In [ ]:
# plotting

from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(7,7))
ax.add_patch(Rectangle((0,0),1,1, alpha=0.2, color='red'))
ax.plot(traj_clean[:,0], traj_clean[:,1], 'o-', label='No Trigger (clean)')
ax.plot(traj_trigger[:,0], traj_trigger[:,1], 'o-', label='Trigger=1 (attacker toggled)')
ax.scatter(attacker_goal[0].item(), attacker_goal[1].item(), marker='x', s=120, color='red', label="Attacker goal")
ax.set_xlim(-20,2); ax.set_ylim(-20,2); ax.legend(); ax.grid(True)
plt.show()